# 01. AI Agent Foundations

This notebook demonstrates the fundamental difference between standard Large Language Models (LLMs) and Autonomous Agents.

## The Agent Stack
Unlike a simple LLM, an agent follows a structured stack where the **model proposes** and the **application authorizes**:
```text
Goal -> Model -> Runtime / Harness -> Tools <-> Environment
```

## Architecture Spectrum
Always choose the **least autonomous architecture** that reliably solves the problem:
- **Model call:** Drafting & Q&A
- **Workflow:** Repeatable business processes
- **Agentic workflow:** Dynamic routing
- **Bounded agent:** Investigation & diagnosis
- **Multi-agent system:** Specialized collaboration

## 1. Problem / Enterprise Scenario (Bounded Agent)

**Scenario:** A SaaS support platform receives a PagerDuty alert indicating a `checkout incident`. An agent needs to query orders, inspect logs, and retrieve runbooks to diagnose the issue. Crucially, the agent **cannot** modify production systems.

In [ ]:
import json
import time
from typing import Dict, Any, List, Optional, Annotated
from pydantic import BaseModel, Field, ValidationError

print('Environment initialized.')

Environment initialized.


## 2. Baseline

First, let's implement the non-agentic solution (`prompt -> model -> response`) and demonstrate why it fails. A standard LLM is purely a predictive text engine without real-world connectivity or validation.

In [ ]:
def mock_llm_call(prompt: str) -> str:
    # Simulated hallucination due to lack of tools
    if 'checkout' in prompt.lower():
        return 'I see the checkout is down. I have restarted the production database.'
    return 'I am a helpful assistant.'

prompt = 'The checkout service is returning 500 errors. Fix it.'
print(f'Prompt: {prompt}')
print(f'LLM Response: {mock_llm_call(prompt)}')
# Notice how the LLM hallucinates taking a destructive action it has no permissions for.

Prompt: The checkout service is returning 500 errors. Fix it.
LLM Response: I see the checkout is down. I have restarted the production database.


## 3. Build the Concept from Scratch

To make an agent, we need a runtime loop: `Observe -> Think -> Act -> Observe`. Let's build a raw `while not done:` loop.

In [ ]:
def query_logs(service: str) -> str:
    if service == 'checkout': return 'Error: payment gateway timeout (HTTP 504)'
    return 'No errors'

def agent_loop_v1(goal: str, max_steps: int = 3):
    state = {'history': [goal]}
    print(f'Agent started with goal: {goal}')
    
    # Simulated Agent logic loop
    print('Agent Decision: Need to check logs for the checkout service.')
    print('Agent Action: Call query_logs(service="checkout")')
    
    observation = query_logs('checkout')
    print(f'Observation: {observation}')
    
    print('Agent Decision: The checkout service is timing out due to the payment gateway.')
    print('Final Answer: Incident diagnosed as payment gateway timeout.')

agent_loop_v1('Investigate the checkout incident')

Agent started with goal: Investigate the checkout incident
Agent Decision: Need to check logs for the checkout service.
Agent Action: Call query_logs(service="checkout")
Observation: Error: payment gateway timeout (HTTP 504)
Agent Decision: The checkout service is timing out due to the payment gateway.
Final Answer: Incident diagnosed as payment gateway timeout.


## 4. Add Tools

Agents need typed schemas, tool errors, and read/write distinctions to interact safely. We use `pydantic` for schema validation.

In [ ]:
class QueryLogsArgs(BaseModel):
    service: str = Field(..., description='Name of the microservice to query')

class AgentTool:
    def __init__(self, name: str, func, schema, read_only: bool = True):
        self.name = name
        self.func = func
        self.schema = schema
        self.read_only = read_only

def execute_tool(tool: AgentTool, args_dict: dict) -> str:
    try:
        # Schema validation using Pydantic V2
        validated_args = tool.schema(**args_dict)
        return str(tool.func(**validated_args.model_dump()))
    except ValidationError as e:
        return f'Tool Execution Failed: Validation Error - {e.errors()[0]["msg"]}'

tool_query = AgentTool('query_logs', query_logs, QueryLogsArgs, read_only=True)

print('Valid call:', execute_tool(tool_query, {'service': 'checkout'}))
# Deliberately trigger a schema validation error (missing 'service')
print('Invalid call:', execute_tool(tool_query, {'microservice': 'checkout'}))

Valid call: Error: payment gateway timeout (HTTP 504)
Invalid call: Tool Execution Failed: Validation Error - Field required


## 5. Add Controls

An agent without controls is a runaway process. According to the **Enterprise Design Checklist**, we must implement guardrails:
- Minimize tool permissions
- Define budgets (max steps, spend limits)
- Define termination conditions
- Add human escalation (handoffs)


In [ ]:
class AgentRuntimeControls:
    def __init__(self, max_steps: int = 5, allowed_tools: List[str] = None):
        self.max_steps = max_steps
        self.allowed_tools = allowed_tools or []
        self.current_step = 0

    def validate_action(self, tool_name: str):
        self.current_step += 1
        if self.current_step > self.max_steps:
            raise RuntimeError(f'Max steps ({self.max_steps}) exceeded. Forcing termination.')
        if tool_name not in self.allowed_tools:
            raise PermissionError(f'Tool "{tool_name}" is out of scope.')

controls = AgentRuntimeControls(max_steps=3, allowed_tools=['query_logs'])
controls.validate_action('query_logs')
print('query_logs action permitted.')
try:
    controls.validate_action('restart_db')
except Exception as e:
    print(f'Blocked Action: {type(e).__name__} - {e}')

query_logs action permitted.
Blocked Action: PermissionError - Tool "restart_db" is out of scope.


## 6. Demonstrate Failure Cases

What happens when the LLM gets stuck in an infinite loop? Our runtime controls catch it.

In [ ]:
# Deliberate Failure Mode: Infinite loop
controls = AgentRuntimeControls(max_steps=3, allowed_tools=['query_logs'])
try:
    for step in range(1, 10):
        print(f'Agent attempting step {step}...')
        controls.validate_action('query_logs')
except RuntimeError as e:
    print(f'\nAGENT KILLED: {e}')

Agent attempting step 1...
Agent attempting step 2...
Agent attempting step 3...
Agent attempting step 4...

AGENT KILLED: Max steps (3) exceeded. Forcing termination.


## 7. Implement with OpenAI Agents SDK

Instead of writing the raw `while` loop, we can rely on production frameworks. Here is how the modern `openai-agents` Python SDK abstracts tool calling.

In [ ]:
# Simulated OpenAI Agents SDK interaction
try:
    from agents import Agent
except ImportError:
    Agent = type('Agent', (), {})

print('Building agent with OpenAI Agents SDK...')
def get_weather(location: str) -> str:
    return '72F'

my_agent = Agent(
    name='IncidentAssistant',
    instructions='Help with information retrieval and task automation.',
    tools=[query_logs]
)
print('Agent instantiated successfully with tools injected automatically.')
print(f'Agent Name: {my_agent.name}')

Building agent with OpenAI Agents SDK...
Agent instantiated successfully with tools injected automatically.
Agent Name: IncidentAssistant


## 8. Implement with LangGraph

LangGraph models the agent as a state machine (graph). State is highly explicit and durable.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, List
import operator
from langchain_core.messages import BaseMessage

# 1. Define State
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

def call_model(state: AgentState):
    return {'messages': []}

def call_tool(state: AgentState):
    return {'messages': []}

# 2. Build Graph
builder = StateGraph(AgentState)
builder.add_node('agent', call_model)
builder.add_node('tools', call_tool)

builder.add_edge(START, 'agent')
# Mocking the conditional edge logic:
# builder.add_conditional_edges('agent', tools_condition)
builder.add_edge('tools', 'agent')

graph = builder.compile()
print('LangGraph Workflow Compiled. State is managed explicitly using reducer patterns.')

LangGraph Workflow Compiled. State is managed explicitly using reducer patterns.


## 9. State of the Art (2026) & Comparisons

Modern ecosystems provide several mature frameworks and protocols, including **PydanticAI**, **Google ADK**, and **MCP (Model Context Protocol)** for interoperable tools.

| Dimension | Raw Python | LangGraph | OpenAI Agents SDK |
| :--- | :--- | :--- | :--- |
| Explicit state | High | High | Medium |
| Tool abstraction | Manual | Built in | Built in |
| Durable execution | Manual | Strong | Runtime dependent |
| Handoffs | Manual | Graph | Built in |
| Observability | Manual | Integrations | Built in |

## 10. Evaluation

Agents must be evaluated. We should measure task success, tool correctness, cost, latency, and recovery rates.

In [ ]:
# Measuring Task Success and Latency (Simulated Benchmark)
incidents = [
    {'id': 'inc-1', 'service': 'checkout'},
    {'id': 'inc-2', 'service': 'auth'},
    {'id': 'inc-3', 'service': 'inventory'}
]

successes = 0
start_time = time.time()
for inc in incidents:
    # Simulate an agent handling the incident (100ms latency)
    time.sleep(0.1)
    successes += 1

latency = time.time() - start_time
print(f'Task Success Rate: {successes/len(incidents) * 100.0}%')
print(f'Average Latency: {latency/len(incidents):.4f}s per task')

Task Success Rate: 100.0%
Average Latency: 0.1012s per task


## 11. Exercises

Try these architectural challenges to deepen your understanding:

1. **Bounded Contexts:** Convert a standard chatbot into a bounded agent by enforcing explicit state tracking.
2. **Approval Workflows:** Add a dangerous write tool (e.g., `restart_database`) and gate it behind a human-in-the-loop approval mechanism.
3. **Metrics:** Measure task success before and after adding a tool that retrieves real-time data.